# AHR Sudoku — Pure Neural Notebook

This notebook rebuilds the Sudoku pipeline around a **pure AHRLiteHRM neural reasoner**:

- the dataset is generated once into on-disk `.npy` arrays with Sudoku symmetry augmentation
- training reads those arrays through `numpy.load(..., mmap_mode='r')`
- the model is only the **worker graph + queen transformer** architecture
- validation and test success are measured as **pure exact neural solves**: every digit correct and conflict energy exactly zero

The defaults below are set for a serious A100 run. Adjust dataset size caps if you want a shorter debug cycle.


## Cell 1 — Imports, config, seed, device

This cell sets up the notebook for high-throughput CUDA training: TF32 is enabled, `bfloat16` autocast is used when available, and the default config targets a large A100 run.


In [ ]:
import csv
import json
import os
import random
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from huggingface_hub import hf_hub_download
except ImportError as exc:
    raise ImportError(
        "Install huggingface_hub in this notebook environment with: %pip install huggingface_hub"
    ) from exc


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class CFG:
    seed: int = 42

    # dataset generation
    source_repo: str = "sapientinc/sudoku-extreme"
    dataset_root: str = "data/sudoku_extreme_memmap"
    train_subsample_size: Optional[int] = None
    test_subsample_size: Optional[int] = None
    min_difficulty: Optional[int] = None
    num_aug: int = 9
    force_regenerate: bool = False

    # mmap split sizes after generation
    val_fraction: float = 0.05
    max_train_examples: Optional[int] = 1_000_000
    max_val_examples: Optional[int] = 50_000
    max_test_examples: Optional[int] = None

    # dataloader
    batch_size: int = 256
    num_workers: int = min(16, os.cpu_count() or 8)
    prefetch_factor: int = 4

    # pure AHRLiteHRM
    hidden_dim: int = 192
    num_heads: int = 8
    queen_period: int = 4
    queen_ff_mult: int = 4
    steps: int = 40
    detach_interval: Optional[int] = 10

    rho: float = 0.35
    noise_scale: float = 0.08
    obs_gain: float = 2.2
    peer_gain: float = 1.25
    tau_clip: float = 12.0
    clue_on: float = 12.0
    clue_off: float = -12.0

    # optimization
    epochs: int = 100
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    warmup_epochs: int = 3
    blank_cell_weight: float = 2.0
    final_blank_weight: float = 1.25
    peer_penalty: float = 0.10

    # runtime
    mixed_precision: bool = True
    compile_model: bool = True
    checkpoint_path: str = "checkpoints/ahr_lite_hrm_pure_best.pt"


cfg = CFG()
seed_everything(cfg.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def amp_context():
    if DEVICE.type == "cuda" and cfg.mixed_precision:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(asdict(cfg), indent=2))


## Cell 2 — Generate the `.npy` corpus on disk

This cell mirrors the logic from `build_sudoku_dataset.py`: it downloads the Hugging Face CSV splits, applies Sudoku-preserving symmetry augmentation to the training set, and writes compact `.npy` arrays plus metadata.

The saved encoding matches the offline builder style:

- `all__inputs.npy`: puzzle digits shifted by `+1`, so blank becomes `1`
- `all__labels.npy`: solution digits shifted by `+1`
- metadata and index arrays are written alongside the memmaps


In [ ]:
def shuffle_sudoku(board: np.ndarray, solution: np.ndarray, rng: np.random.Generator):
    digit_map = np.pad(rng.permutation(np.arange(1, 10, dtype=np.uint8)), (1, 0))
    transpose_flag = rng.random() < 0.5

    bands = rng.permutation(3)
    row_perm = np.concatenate([b * 3 + rng.permutation(3) for b in bands])

    stacks = rng.permutation(3)
    col_perm = np.concatenate([s * 3 + rng.permutation(3) for s in stacks])

    mapping = np.array([row_perm[i // 9] * 9 + col_perm[i % 9] for i in range(81)], dtype=np.int64)

    def apply_transform(x: np.ndarray) -> np.ndarray:
        x = x.T if transpose_flag else x
        x = x.reshape(-1)[mapping].reshape(9, 9)
        return digit_map[x]

    return apply_transform(board), apply_transform(solution)


def read_repo_split(
    repo_id: str,
    split_name: str,
    min_difficulty: Optional[int] = None,
    subsample_size: Optional[int] = None,
    seed: int = 42,
):
    csv_path = hf_hub_download(repo_id=repo_id, filename=f"{split_name}.csv", repo_type="dataset")

    inputs = []
    labels = []
    with open(csv_path, newline="") as csvfile:
        reader = csv.reader(csvfile)
        next(reader, None)
        for source, q, a, rating in tqdm(reader, desc=f"read-{split_name}"):
            if min_difficulty is not None and int(rating) < min_difficulty:
                continue
            if len(q) != 81 or len(a) != 81:
                continue
            inputs.append(np.frombuffer(q.replace('.', '0').encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
            labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))

    if not inputs:
        raise RuntimeError(f"No puzzles loaded for split={split_name!r}")

    inputs = np.stack(inputs).astype(np.uint8)
    labels = np.stack(labels).astype(np.uint8)

    if subsample_size is not None and subsample_size < len(inputs):
        rng = np.random.default_rng(seed)
        keep = rng.choice(len(inputs), size=subsample_size, replace=False)
        inputs = inputs[keep]
        labels = labels[keep]

    return inputs, labels


def convert_subset(split_name: str, cfg: CFG):
    inputs, labels = read_repo_split(
        repo_id=cfg.source_repo,
        split_name=split_name,
        min_difficulty=cfg.min_difficulty,
        subsample_size=cfg.train_subsample_size if split_name == "train" else cfg.test_subsample_size,
        seed=cfg.seed + (0 if split_name == "train" else 1),
    )

    num_augments = cfg.num_aug if split_name == "train" else 0
    examples_per_group = 1 + num_augments
    total_examples = len(inputs) * examples_per_group

    encoded_inputs = np.empty((total_examples, 81), dtype=np.uint8)
    encoded_labels = np.empty((total_examples, 81), dtype=np.uint8)
    puzzle_identifiers = np.zeros(total_examples, dtype=np.int32)
    puzzle_indices = np.arange(total_examples + 1, dtype=np.int32)
    group_indices = np.empty(len(inputs) + 1, dtype=np.int32)
    group_indices[0] = 0

    rng = np.random.default_rng(cfg.seed + (0 if split_name == "train" else 10_000))
    write_idx = 0
    for group_idx, (board, solution) in enumerate(tqdm(zip(inputs, labels), total=len(inputs), desc=f"augment-{split_name}")):
        for aug_idx in range(examples_per_group):
            if aug_idx == 0:
                cur_board, cur_solution = board, solution
            else:
                cur_board, cur_solution = shuffle_sudoku(board, solution, rng)

            encoded_inputs[write_idx] = cur_board.reshape(-1).astype(np.uint8) + 1
            encoded_labels[write_idx] = cur_solution.reshape(-1).astype(np.uint8) + 1
            write_idx += 1

        group_indices[group_idx + 1] = write_idx

    split_dir = Path(cfg.dataset_root) / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "seq_len": 81,
        "vocab_size": 11,
        "pad_id": 0,
        "ignore_label_id": 0,
        "blank_identifier_id": 0,
        "num_puzzle_identifiers": 1,
        "total_groups": int(len(inputs)),
        "mean_puzzle_examples": int(examples_per_group),
        "sets": ["all"],
        "storage_encoding": {
            "inputs": "digits shifted by +1 so blank=1",
            "labels": "solution digits shifted by +1",
        },
    }

    with open(split_dir / "dataset.json", "w") as f:
        json.dump(metadata, f, indent=2)

    np.save(split_dir / "all__inputs.npy", encoded_inputs)
    np.save(split_dir / "all__labels.npy", encoded_labels)
    np.save(split_dir / "all__puzzle_identifiers.npy", puzzle_identifiers)
    np.save(split_dir / "all__puzzle_indices.npy", puzzle_indices)
    np.save(split_dir / "all__group_indices.npy", group_indices)

    print(f"Saved {split_name}: {len(inputs):,} source puzzles -> {total_examples:,} examples")


def ensure_numpy_corpus(cfg: CFG):
    expected = [
        Path(cfg.dataset_root) / "train" / "all__inputs.npy",
        Path(cfg.dataset_root) / "train" / "all__labels.npy",
        Path(cfg.dataset_root) / "test" / "all__inputs.npy",
        Path(cfg.dataset_root) / "test" / "all__labels.npy",
    ]
    if not cfg.force_regenerate and all(path.exists() for path in expected):
        print(f"Using existing memmap corpus under {cfg.dataset_root}")
        return

    Path(cfg.dataset_root).mkdir(parents=True, exist_ok=True)
    convert_subset("train", cfg)
    convert_subset("test", cfg)

    with open(Path(cfg.dataset_root) / "identifiers.json", "w") as f:
        json.dump(["<blank>"], f, indent=2)


ensure_numpy_corpus(cfg)


## Cell 3 — Memory-mapped dataset and loaders

The training loop never materializes the full corpus in RAM. Each worker opens the `.npy` arrays with `mmap_mode='r'`, decodes a single puzzle/solution pair, and ships it directly into pinned host memory for the GPU.

The train/validation split is done **by original puzzle group**, not by augmented example. That prevents symmetry-augmented siblings of the same puzzle from leaking across splits and inflating validation accuracy.


In [ ]:
class MemoryMappedSudokuDataset(Dataset):
    def __init__(self, split_dir: str | Path, indices: Optional[np.ndarray] = None):
        self.split_dir = Path(split_dir)
        self.inputs_path = self.split_dir / "all__inputs.npy"
        self.labels_path = self.split_dir / "all__labels.npy"
        self.indices = None if indices is None else np.asarray(indices, dtype=np.int64)

        probe = np.load(self.inputs_path, mmap_mode="r")
        self.total_examples = int(probe.shape[0])
        del probe

        self._inputs = None
        self._labels = None

    def _ensure_open(self):
        if self._inputs is None:
            self._inputs = np.load(self.inputs_path, mmap_mode="r")
            self._labels = np.load(self.labels_path, mmap_mode="r")

    def __len__(self):
        return self.total_examples if self.indices is None else int(len(self.indices))

    def __getitem__(self, idx):
        self._ensure_open()
        if self.indices is not None:
            idx = int(self.indices[idx])

        puzzle = np.asarray(self._inputs[idx], dtype=np.int64) - 1
        solution = np.asarray(self._labels[idx], dtype=np.int64) - 1
        return torch.as_tensor(puzzle.copy(), dtype=torch.long), torch.as_tensor(solution.copy(), dtype=torch.long)


def build_group_preserving_train_val_indices(train_dir: str | Path, cfg: CFG):
    train_dir = Path(train_dir)
    group_boundaries = np.load(train_dir / "all__group_indices.npy", mmap_mode="r")
    total_examples = int(group_boundaries[-1])
    num_groups = int(len(group_boundaries) - 1)

    rng = np.random.default_rng(cfg.seed)
    group_order = rng.permutation(num_groups)

    target_val_examples = max(1, int(round(total_examples * cfg.val_fraction)))
    if cfg.max_val_examples is not None:
        target_val_examples = min(target_val_examples, cfg.max_val_examples)

    val_groups = []
    train_groups = []
    val_examples = 0
    for group_id in group_order:
        start = int(group_boundaries[group_id])
        end = int(group_boundaries[group_id + 1])
        group_size = end - start

        if val_examples < target_val_examples:
            val_groups.append(group_id)
            val_examples += group_size
        else:
            train_groups.append(group_id)

    def groups_to_indices(group_ids, example_cap: Optional[int] = None):
        chunks = []
        count = 0
        for group_id in group_ids:
            start = int(group_boundaries[group_id])
            end = int(group_boundaries[group_id + 1])
            group_idx = np.arange(start, end, dtype=np.int64)
            group_size = len(group_idx)

            if example_cap is not None and count > 0 and count + group_size > example_cap:
                break

            chunks.append(group_idx)
            count += group_size

            if example_cap is not None and count >= example_cap:
                break

        if not chunks:
            return np.empty(0, dtype=np.int64)
        return np.concatenate(chunks)

    train_idx = groups_to_indices(train_groups, cfg.max_train_examples)
    val_idx = groups_to_indices(val_groups, cfg.max_val_examples)

    print(
        f"group split: train_groups={len(train_groups):,} val_groups={len(val_groups):,} "
        f"train_examples={len(train_idx):,} val_examples={len(val_idx):,}"
    )
    return np.sort(train_idx), np.sort(val_idx)


def make_dataloaders(cfg: CFG):
    train_dir = Path(cfg.dataset_root) / "train"
    test_dir = Path(cfg.dataset_root) / "test"

    train_probe = np.load(train_dir / "all__inputs.npy", mmap_mode="r")
    test_probe = np.load(test_dir / "all__inputs.npy", mmap_mode="r")
    total_test_examples = int(test_probe.shape[0])
    del train_probe, test_probe

    train_idx, val_idx = build_group_preserving_train_val_indices(train_dir, cfg)
    test_idx = np.arange(total_test_examples, dtype=np.int64)
    if cfg.max_test_examples is not None:
        test_idx = test_idx[:cfg.max_test_examples]

    train_ds = MemoryMappedSudokuDataset(train_dir, train_idx)
    val_ds = MemoryMappedSudokuDataset(train_dir, val_idx)
    test_ds = MemoryMappedSudokuDataset(test_dir, test_idx)

    loader_kwargs = {
        "batch_size": cfg.batch_size,
        "num_workers": cfg.num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "persistent_workers": cfg.num_workers > 0,
    }
    if cfg.num_workers > 0:
        loader_kwargs["prefetch_factor"] = cfg.prefetch_factor

    generator = torch.Generator()
    generator.manual_seed(cfg.seed)

    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, generator=generator, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)

    print(f"train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}")
    return train_loader, val_loader, test_loader


## Cell 4 — Peer graph, energy, and pure exact metrics

`conflict_energy` is the only structural validity check used for evaluation. A validation example counts as solved only when the network predicts all 81 digits correctly **and** the resulting board has zero clue or peer conflicts.


In [ ]:
def build_peer_graph() -> torch.Tensor:
    adj = torch.zeros(81, 81, dtype=torch.float32)
    for i in range(81):
        r1, c1 = divmod(i, 9)
        b1r, b1c = r1 // 3, c1 // 3
        for j in range(81):
            if i == j:
                continue
            r2, c2 = divmod(j, 9)
            same_row = r1 == r2
            same_col = c1 == c2
            same_box = (b1r == r2 // 3) and (b1c == c2 // 3)
            if same_row or same_col or same_box:
                adj[i, j] = 1.0
    return adj


PEER_GRAPH = build_peer_graph()
PEER_EDGES = torch.triu(PEER_GRAPH, diagonal=1).nonzero(as_tuple=False)
DEGREE = PEER_GRAPH.sum(dim=-1, keepdim=True).clamp_min(1.0)

print("Peer graph shape:", tuple(PEER_GRAPH.shape))
print("Unique peer edges:", int(PEER_EDGES.shape[0]))


def pretty_board(board) -> str:
    board = np.asarray(board).reshape(9, 9)
    rows = []
    for r in range(9):
        parts = []
        for c in range(9):
            value = int(board[r, c])
            parts.append("." if value == 0 else str(value))
            if c in (2, 5):
                parts.append("|")
        rows.append(" ".join(parts))
        if r in (2, 5):
            rows.append("-" * 25)
    return "\n".join(rows)


def logits_to_digits(logits: torch.Tensor) -> torch.Tensor:
    return logits.argmax(dim=-1) + 1


def conflict_energy(pred_board: torch.Tensor, puzzle_board: torch.Tensor) -> torch.Tensor:
    if pred_board.ndim == 1:
        pred_board = pred_board.unsqueeze(0)
    if puzzle_board.ndim == 1:
        puzzle_board = puzzle_board.unsqueeze(0)

    edge_index = PEER_EDGES.to(pred_board.device)
    u = edge_index[:, 0]
    v = edge_index[:, 1]

    peer_conflicts = (pred_board[:, u] == pred_board[:, v]).sum(dim=1)
    clue_conflicts = ((puzzle_board > 0) & (pred_board != puzzle_board)).sum(dim=1)
    return peer_conflicts + clue_conflicts


@torch.no_grad()
def compute_batch_metrics(logits_traj: torch.Tensor, puzzle_digits: torch.Tensor, target_digits: torch.Tensor):
    final_pred = logits_to_digits(logits_traj[-1])
    energy = conflict_energy(final_pred, puzzle_digits)

    all_digits_correct = (final_pred == target_digits).all(dim=1)
    zero_energy = energy == 0
    pure_exact = all_digits_correct & zero_energy

    return {
        "cell_accuracy": float((final_pred == target_digits).float().mean().item()),
        "zero_energy_rate": float(zero_energy.float().mean().item()),
        "pure_exact_rate": float(pure_exact.float().mean().item()),
        "mean_energy": float(energy.float().mean().item()),
    }


## Cell 5 — Pure AHRLiteHRM model

This is the pure neural solver. There is no deterministic repair stage, no backtracking, and no C++ fallback. Reasoning happens only through repeated worker graph updates plus the slower queen transformer over row/column/box region tokens.


In [ ]:
class WorkerLayer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * d_model + 9, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, h: torch.Tensor, p: torch.Tensor, adj: torch.Tensor, deg: torch.Tensor):
        peer_h = torch.einsum("ij,bjd->bid", adj, h) / deg.unsqueeze(0)
        peer_p = torch.einsum("ij,bjd->bid", adj, p) / deg.unsqueeze(0)
        out = self.net(torch.cat([h, peer_h, peer_p], dim=-1))
        return self.norm(h + out)


class QueenRegionLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int = 4):
        super().__init__()
        ff_hidden = ff_mult * d_model
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_hidden),
            nn.GELU(),
            nn.Linear(ff_hidden, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, region_tokens: torch.Tensor):
        attn_out, _ = self.attn(region_tokens, region_tokens, region_tokens, need_weights=False)
        x = self.norm1(region_tokens + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class AHRLiteHRM(nn.Module):
    def __init__(
        self,
        adjacency: torch.Tensor,
        hidden_dim: int = 192,
        num_heads: int = 8,
        queen_period: int = 4,
        queen_ff_mult: int = 4,
        rho: float = 0.35,
        noise_scale: float = 0.08,
        obs_gain: float = 2.2,
        peer_gain: float = 1.25,
        tau_clip: float = 12.0,
        clue_on: float = 12.0,
        clue_off: float = -12.0,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.queen_period = queen_period
        self.rho = rho
        self.noise_scale = noise_scale
        self.obs_gain = obs_gain
        self.peer_gain = peer_gain
        self.tau_clip = tau_clip
        self.clue_on = clue_on
        self.clue_off = clue_off

        self.register_buffer("adj", adjacency.float())
        self.register_buffer("deg", adjacency.float().sum(dim=-1, keepdim=True).clamp_min(1.0))

        row_ids = torch.tensor([i // 9 for i in range(81)], dtype=torch.long)
        col_ids = torch.tensor([i % 9 for i in range(81)], dtype=torch.long)
        box_ids = torch.tensor([3 * ((i // 9) // 3) + ((i % 9) // 3) for i in range(81)], dtype=torch.long)
        self.register_buffer("row_ids", row_ids)
        self.register_buffer("col_ids", col_ids)
        self.register_buffer("box_ids", box_ids)

        self.blank_tau = nn.Parameter(torch.zeros(1, 81, 9))
        self.pos = nn.Parameter(torch.randn(1, 81, hidden_dim) * 0.02)

        self.encoder = nn.Sequential(
            nn.Linear(19, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.worker = WorkerLayer(hidden_dim)
        self.queen = QueenRegionLayer(hidden_dim, num_heads, ff_mult=queen_ff_mult)

        self.global_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.z_to_cells = nn.Linear(hidden_dim, hidden_dim, bias=False)

        update_mid = max(hidden_dim // 2, 64)
        self.delta_head = nn.Sequential(
            nn.Linear(hidden_dim, update_mid),
            nn.GELU(),
            nn.Linear(update_mid, 9),
        )

    def make_clue_tensors(self, x_digits: torch.Tensor):
        clue_mask = (x_digits > 0).float().unsqueeze(-1)
        safe_digits = x_digits.clamp(min=1) - 1
        clue_onehot = F.one_hot(safe_digits, num_classes=9).float() * clue_mask

        clue_logits = torch.full_like(clue_onehot, self.clue_off)
        clue_logits = clue_logits + clue_onehot * (self.clue_on - self.clue_off)
        return clue_onehot, clue_mask, clue_logits

    def build_region_tokens(self, h: torch.Tensor):
        batch_size, _, dim = h.shape
        grid = h.view(batch_size, 9, 9, dim)

        row_tokens = grid.mean(dim=2)
        col_tokens = grid.mean(dim=1)
        box_tokens = (
            grid.view(batch_size, 3, 3, 3, 3, dim)
            .permute(0, 1, 3, 2, 4, 5)
            .contiguous()
            .view(batch_size, 9, 9, dim)
            .mean(dim=2)
        )
        return torch.cat([row_tokens, col_tokens, box_tokens], dim=1)

    def broadcast_region_tokens(self, region_tokens: torch.Tensor):
        row_tok = region_tokens[:, 0:9, :]
        col_tok = region_tokens[:, 9:18, :]
        box_tok = region_tokens[:, 18:27, :]

        row_ctx = row_tok[:, self.row_ids, :]
        col_ctx = col_tok[:, self.col_ids, :]
        box_ctx = box_tok[:, self.box_ids, :]
        return (row_ctx + col_ctx + box_ctx) / 3.0

    def forward(
        self,
        x_digits: torch.Tensor,
        steps: int = 40,
        use_noise: bool = True,
        detach_interval: Optional[int] = None,
    ):
        batch_size = x_digits.shape[0]
        clue_onehot, clue_mask, clue_logits = self.make_clue_tensors(x_digits)

        tau = self.blank_tau.expand(batch_size, -1, -1).clone()
        tau = tau * (1.0 - clue_mask) + clue_logits * clue_mask

        global_state = torch.zeros(batch_size, self.hidden_dim, device=x_digits.device)
        queen_context = torch.zeros(batch_size, 81, self.hidden_dim, device=x_digits.device)

        logits_traj = []
        probs_traj = []
        conf_traj = []

        for t in range(steps):
            probs = F.softmax(tau, dim=-1)

            h = torch.cat([probs, clue_onehot, clue_mask], dim=-1)
            h = self.encoder(h)
            h = h + self.pos + self.z_to_cells(global_state).unsqueeze(1)
            h = self.worker(h, probs, self.adj, self.deg)

            if t == 0 or (t % self.queen_period == 0):
                region_tokens = self.build_region_tokens(h)
                region_tokens = self.queen(region_tokens)
                queen_context = self.broadcast_region_tokens(region_tokens)

            h_eff = h + queen_context
            delta_tau = self.delta_head(h_eff)

            obs_err = self.obs_gain * (clue_mask * (clue_onehot - probs))
            peer_same = torch.einsum("ij,bjd->bid", self.adj, probs) / self.deg.unsqueeze(0)

            if use_noise:
                anneal = 1.0 - (t / max(steps - 1, 1))
                scout = torch.randn_like(tau) * (self.noise_scale * anneal)
            else:
                scout = 0.0

            proposal = tau + delta_tau + obs_err - self.peer_gain * peer_same + scout
            tau = torch.clamp(
                (1.0 - self.rho) * tau + self.rho * proposal,
                min=-self.tau_clip,
                max=self.tau_clip,
            )
            tau = tau * (1.0 - clue_mask) + clue_logits * clue_mask

            global_state = self.global_pool(h_eff.mean(dim=1))

            logits_traj.append(tau)
            probs_traj.append(F.softmax(tau, dim=-1))
            conf_traj.append(F.softmax(tau, dim=-1).max(dim=-1).values)

            if detach_interval is not None and (t + 1) % detach_interval == 0:
                tau = tau.detach()
                queen_context = queen_context.detach()
                global_state = global_state.detach()

        return (
            torch.stack(logits_traj, dim=0),
            torch.stack(probs_traj, dim=0),
            torch.stack(conf_traj, dim=0),
        )


## Cell 6 — Loss, validation, checkpointing, and plots

The loss is still pure neural supervision. It combines:

- deep supervision across the full trajectory
- extra pressure on blank cells at the final step
- a lightweight peer-conflict regularizer on the final distribution

The checkpoint criterion is the validation `pure_exact_rate`.


In [ ]:
def neural_reasoning_loss(logits_traj: torch.Tensor, puzzle_digits: torch.Tensor, target_digits: torch.Tensor, cfg: CFG):
    steps, batch_size, cells, num_classes = logits_traj.shape
    targets = target_digits - 1
    repeated_targets = targets.unsqueeze(0).expand(steps, -1, -1)

    ce_per_cell = F.cross_entropy(
        logits_traj.reshape(steps * batch_size * cells, num_classes),
        repeated_targets.reshape(-1),
        reduction="none",
    ).view(steps, batch_size, cells)

    step_weights = torch.linspace(0.25, 1.25, steps, device=logits_traj.device, dtype=logits_traj.dtype)
    step_weights = step_weights / step_weights.sum()

    cell_weights = 1.0 + cfg.blank_cell_weight * (puzzle_digits == 0).float()
    trajectory_ce = (ce_per_cell * step_weights.view(steps, 1, 1) * cell_weights.unsqueeze(0)).sum()
    trajectory_ce = trajectory_ce / cell_weights.sum().clamp_min(1.0)

    final_ce_per_cell = F.cross_entropy(
        logits_traj[-1].reshape(batch_size * cells, num_classes),
        targets.reshape(-1),
        reduction="none",
    ).view(batch_size, cells)
    blank_mask = (puzzle_digits == 0).float()
    final_blank_ce = (final_ce_per_cell * blank_mask).sum() / blank_mask.sum().clamp_min(1.0)

    final_probs = F.softmax(logits_traj[-1], dim=-1)
    peer_same = torch.einsum("ij,bjd->bid", PEER_GRAPH.to(logits_traj.device), final_probs)
    peer_same = peer_same / DEGREE.to(logits_traj.device).unsqueeze(0)
    peer_reg = ((final_probs * peer_same).sum(dim=-1) * blank_mask).sum() / blank_mask.sum().clamp_min(1.0)

    loss = trajectory_ce + cfg.final_blank_weight * final_blank_ce + cfg.peer_penalty * peer_reg
    parts = {
        "trajectory_ce": float(trajectory_ce.detach().item()),
        "final_blank_ce": float(final_blank_ce.detach().item()),
        "peer_reg": float(peer_reg.detach().item()),
        "loss": float(loss.detach().item()),
    }
    return loss, parts


def make_model(cfg: CFG):
    model = AHRLiteHRM(
        adjacency=PEER_GRAPH.to(DEVICE),
        hidden_dim=cfg.hidden_dim,
        num_heads=cfg.num_heads,
        queen_period=cfg.queen_period,
        queen_ff_mult=cfg.queen_ff_mult,
        rho=cfg.rho,
        noise_scale=cfg.noise_scale,
        obs_gain=cfg.obs_gain,
        peer_gain=cfg.peer_gain,
        tau_clip=cfg.tau_clip,
        clue_on=cfg.clue_on,
        clue_off=cfg.clue_off,
    ).to(DEVICE)
    return model


def build_optimizer(model: nn.Module, cfg: CFG):
    kwargs = {
        "params": model.parameters(),
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
    }
    if DEVICE.type == "cuda":
        try:
            kwargs["fused"] = True
        except Exception:
            pass
    return torch.optim.AdamW(**kwargs)


def build_scheduler(optimizer, cfg: CFG):
    if cfg.warmup_epochs <= 0:
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.2,
        end_factor=1.0,
        total_iters=cfg.warmup_epochs,
    )
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, cfg.epochs - cfg.warmup_epochs),
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[cfg.warmup_epochs],
    )


def accumulate_metrics(store: dict, batch_metrics: dict, batch_size: int):
    for key, value in batch_metrics.items():
        store[key] += value * batch_size


def finalize_metrics(store: dict, total_examples: int):
    return {key: value / max(total_examples, 1) for key, value in store.items()}


def train_one_epoch(model, loader, optimizer, cfg: CFG):
    model.train()
    totals = {"loss": 0.0, "cell_accuracy": 0.0, "zero_energy_rate": 0.0, "pure_exact_rate": 0.0, "mean_energy": 0.0}
    total_examples = 0

    pbar = tqdm(loader, desc="train", leave=False)
    for puzzle_digits, target_digits in pbar:
        puzzle_digits = puzzle_digits.to(DEVICE, non_blocking=True)
        target_digits = target_digits.to(DEVICE, non_blocking=True)
        batch_size = puzzle_digits.shape[0]

        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            logits_traj, _, _ = model(
                puzzle_digits,
                steps=cfg.steps,
                use_noise=True,
                detach_interval=cfg.detach_interval,
            )
            loss, loss_parts = neural_reasoning_loss(logits_traj, puzzle_digits, target_digits, cfg)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        metrics = compute_batch_metrics(logits_traj.detach(), puzzle_digits, target_digits)
        metrics["loss"] = float(loss.detach().item())
        accumulate_metrics(totals, metrics, batch_size)
        total_examples += batch_size

        pbar.set_postfix(
            loss=f"{loss_parts['loss']:.4f}",
            pure=f"{metrics['pure_exact_rate']:.3f}",
            energy=f"{metrics['mean_energy']:.2f}",
        )

    return finalize_metrics(totals, total_examples)


@torch.no_grad()
def evaluate(model, loader, cfg: CFG, desc: str = "eval"):
    model.eval()
    totals = {"loss": 0.0, "cell_accuracy": 0.0, "zero_energy_rate": 0.0, "pure_exact_rate": 0.0, "mean_energy": 0.0}
    total_examples = 0

    pbar = tqdm(loader, desc=desc, leave=False)
    for puzzle_digits, target_digits in pbar:
        puzzle_digits = puzzle_digits.to(DEVICE, non_blocking=True)
        target_digits = target_digits.to(DEVICE, non_blocking=True)
        batch_size = puzzle_digits.shape[0]

        with amp_context():
            logits_traj, _, _ = model(
                puzzle_digits,
                steps=cfg.steps,
                use_noise=False,
                detach_interval=None,
            )
            loss, _ = neural_reasoning_loss(logits_traj, puzzle_digits, target_digits, cfg)

        metrics = compute_batch_metrics(logits_traj, puzzle_digits, target_digits)
        metrics["loss"] = float(loss.detach().item())
        accumulate_metrics(totals, metrics, batch_size)
        total_examples += batch_size

    return finalize_metrics(totals, total_examples)


def fit(model, train_loader, val_loader, cfg: CFG):
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)
    checkpoint_path = Path(cfg.checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    history = []
    best_score = float("-inf")
    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model

    for epoch in range(1, cfg.epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, cfg)
        val_metrics = evaluate(model, val_loader, cfg, desc="val")
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        score = (
            1000.0 * val_metrics["pure_exact_rate"]
            + 10.0 * val_metrics["zero_energy_rate"]
            - val_metrics["mean_energy"]
        )
        if score > best_score:
            best_score = score
            torch.save(
                {
                    "model_state_dict": {k: v.detach().cpu().clone() for k, v in model_to_save.state_dict().items()},
                    "cfg": asdict(cfg),
                    "history": history,
                },
                checkpoint_path,
            )

        print(
            f"Epoch {epoch:03d} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_pure_exact={val_metrics['pure_exact_rate']:.4f} | "
            f"val_zero_energy={val_metrics['zero_energy_rate']:.4f} | "
            f"val_mean_energy={val_metrics['mean_energy']:.3f}"
        )

    with open(checkpoint_path.with_suffix(".history.json"), "w") as f:
        json.dump(history, f, indent=2)

    payload = torch.load(checkpoint_path, map_location=DEVICE)
    model_to_save.load_state_dict(payload["model_state_dict"])
    return history


def plot_history(history):
    epochs = [row["epoch"] for row in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="val")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(epochs, [row["train_pure_exact_rate"] for row in history], label="train")
    axes[1].plot(epochs, [row["val_pure_exact_rate"] for row in history], label="val")
    axes[1].set_title("Pure Exact Rate")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    axes[2].plot(epochs, [row["train_mean_energy"] for row in history], label="train")
    axes[2].plot(epochs, [row["val_mean_energy"] for row in history], label="val")
    axes[2].set_title("Conflict Energy")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


## Cell 7 — Build loaders, compile the model, train, and test

This is the end-to-end training entry point. The first epoch is usually slower because `torch.compile` needs to trace and optimize the graph.


In [ ]:
train_loader, val_loader, test_loader = make_dataloaders(cfg)

seed_everything(cfg.seed)
base_model = make_model(cfg)
print(f"Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

train_model = base_model
if cfg.compile_model and DEVICE.type == "cuda":
    try:
        train_model = torch.compile(base_model, mode="max-autotune")
        print("Model compiled with torch.compile(mode='max-autotune').")
    except Exception as exc:
        print("torch.compile failed; continuing with eager mode:", exc)
        train_model = base_model

history = fit(train_model, train_loader, val_loader, cfg)
plot_history(history)

best_payload = torch.load(cfg.checkpoint_path, map_location=DEVICE)
model = make_model(cfg)
model.load_state_dict(best_payload["model_state_dict"])
model.eval()

test_metrics = evaluate(model, test_loader, cfg, desc="test")
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))


## Cell 8 — Inspect a few pure neural predictions

This final cell prints sample boards from the held-out test set so you can inspect raw neural outputs, conflict energy, and whether each example counted as a pure exact solve.


In [ ]:
@torch.no_grad()
def solve_batch(model, puzzle_digits: torch.Tensor, cfg: CFG):
    model.eval()
    logits_traj, _, conf_traj = model(
        puzzle_digits.to(DEVICE),
        steps=cfg.steps,
        use_noise=False,
        detach_interval=None,
    )
    pred = logits_to_digits(logits_traj[-1]).cpu()
    conf = conf_traj[-1].cpu()
    energy = conflict_energy(pred, puzzle_digits.cpu()).cpu()
    return pred, conf, energy


def show_examples(model, loader, cfg: CFG, count: int = 3):
    puzzle_digits, target_digits = next(iter(loader))
    puzzle_digits = puzzle_digits[:count]
    target_digits = target_digits[:count]
    pred_digits, conf, energy = solve_batch(model, puzzle_digits, cfg)

    for idx in range(puzzle_digits.shape[0]):
        pure_exact = bool(torch.equal(pred_digits[idx], target_digits[idx]) and energy[idx].item() == 0)
        print(f"Example {idx} | pure_exact={pure_exact} | energy={int(energy[idx].item())} | mean_conf={float(conf[idx].mean().item()):.4f}")
        print("Puzzle:")
        print(pretty_board(puzzle_digits[idx].numpy()))
        print("\nPrediction:")
        print(pretty_board(pred_digits[idx].numpy()))
        print("\nTarget:")
        print(pretty_board(target_digits[idx].numpy()))
        print("\n" + "=" * 72 + "\n")


show_examples(model, test_loader, cfg, count=3)
